# Messy Mashup - Music Genre Classification
## Training Notebook
Robust genre classification under noisy mixing conditions.

**Models:** XGBoost (Classical ML) | CNN (From Scratch) | CRNN (CNN+GRU) | AST (Pretrained Transformer)

**Metric:** Macro F1 Score across 10 genres

In [ ]:
# ============================================================
# Cell 1: Install Dependencies
# ============================================================
!pip install -q librosa audiomentations wandb transformers xgboost soundfile

In [ ]:
# ============================================================
# Cell 2: Imports
# ============================================================
import os
import random
import warnings
import numpy as np
import pandas as pd
import librosa
import librosa.display
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from pathlib import Path
from tqdm.auto import tqdm
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import wandb

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# ============================================================
# Cell 3: Configuration
# ============================================================
class CFG:
    seed = 42
    base_path = '/kaggle/input/messy-mashup'  # adjust if needed
    sr = 22050
    duration = 10  # seconds
    n_mels = 128
    n_fft = 2048
    hop_length = 512
    fmax = 8000
    img_size = (128, 216)  # (n_mels, time_frames)

    n_folds = 5
    epochs = 35
    batch_size = 32
    lr = 1e-3
    weight_decay = 1e-4
    num_workers = 2

    genres = ["blues", "classical", "country", "disco", "hiphop",
              "jazz", "metal", "pop", "reggae", "rock"]
    genre2idx = {g: i for i, g in enumerate(genres)}
    idx2genre = {i: g for g, i in genre2idx.items()}

    # Augmentation params
    n_mashups_per_genre = 300  # synthetic mashups per genre
    noise_prob = 0.7
    max_noise_snr = 15  # dB

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG.seed)

In [ ]:
# ============================================================
# Cell 4: W&B Setup
# ============================================================
# Replace with your roll number
W_PROJECT = "YourRollNo-t12026"

wandb.login()  # paste your API key when prompted

In [ ]:
# ============================================================
# Cell 5: Explore Dataset Structure
# ============================================================
stems_path = Path(CFG.base_path) / 'genres_stems'
mashups_path = Path(CFG.base_path) / 'mashups'
noise_path = Path(CFG.base_path) / 'ESC-50-master' / 'audio'

print("Genres:", os.listdir(stems_path))
sample_genre_dir = stems_path / 'blues'
print("Sample genre folder:", os.listdir(sample_genre_dir)[:5])
print("Sample song stems:", os.listdir(sample_genre_dir / os.listdir(sample_genre_dir)[0]))
print(f"Test mashups count: {len(os.listdir(mashups_path))}")
print(f"Noise clips count: {len(os.listdir(noise_path))}")

test_df = pd.read_csv(Path(CFG.base_path) / 'test.csv')
sample_sub = pd.read_csv(Path(CFG.base_path) / 'sample_submission.csv')
print(f"\nTest samples: {len(test_df)}")
print(test_df.head())
print(f"\nSample submission:")
print(sample_sub.head())

In [ ]:
# ============================================================
# Cell 6: EDA - Data Exploration
# ============================================================

# 1. Class distribution
genre_counts = {g: len(list((stems_path / g).iterdir())) for g in CFG.genres}
plt.figure(figsize=(10, 4))
plt.bar(genre_counts.keys(), genre_counts.values(), color='steelblue')
plt.title('Songs per Genre'); plt.ylabel('Count'); plt.xticks(rotation=45)
plt.tight_layout(); plt.show()

# 2. Audio duration analysis
print("\nSample audio durations (seconds):")
for genre in CFG.genres[:3]:
    song_dir = list((stems_path / genre).iterdir())[0]
    for stem in ['drums.wav', 'vocals.wav', 'bass.wav', 'others.wav']:
        fpath = song_dir / stem
        if fpath.exists():
            y, sr_ = librosa.load(fpath, sr=None)
            print(f"  {genre}/{song_dir.name}/{stem}: {len(y)/sr_:.2f}s, sr={sr_}")

# 3. Visualize spectrograms per genre
fig, axes = plt.subplots(2, 5, figsize=(20, 6))
for i, genre in enumerate(CFG.genres):
    song_dir = list((stems_path / genre).iterdir())[0]
    stem_file = song_dir / 'others.wav'
    if stem_file.exists():
        y, _ = librosa.load(stem_file, sr=CFG.sr, duration=CFG.duration)
    else:
        y = np.zeros(CFG.sr * CFG.duration)
    mel = librosa.feature.melspectrogram(y=y, sr=CFG.sr, n_mels=CFG.n_mels)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    ax = axes[i // 5, i % 5]
    librosa.display.specshow(mel_db, sr=CFG.sr, hop_length=CFG.hop_length, ax=ax)
    ax.set_title(genre, fontsize=10); ax.set_xlabel(''); ax.set_ylabel('')
plt.suptitle('Mel Spectrograms by Genre (others.wav)'); plt.tight_layout(); plt.show()

# 4. ESC-50 noise category distribution
esc_meta_path = Path(CFG.base_path) / 'ESC-50-master' / 'meta' / 'esc50.csv'
if esc_meta_path.exists():
    esc_meta = pd.read_csv(esc_meta_path)
    print(f"\nESC-50: {len(esc_meta)} clips, {esc_meta['category'].nunique()} categories")
    print(esc_meta['category'].value_counts().head(10))

# 5. Test mashup duration check
sample_mashup = list(mashups_path.glob('*.wav'))[0]
y_test_sample, _ = librosa.load(sample_mashup, sr=None)
print(f"\nSample test mashup: {sample_mashup.name}, duration: {len(y_test_sample)/22050:.2f}s")

In [ ]:
# ============================================================
# Cell 7: Audio Utilities
# ============================================================

def load_audio(path, sr=CFG.sr, duration=CFG.duration):
    """Load audio file, pad/trim to fixed duration."""
    try:
        y, _ = librosa.load(path, sr=sr, duration=duration, mono=True)
    except Exception:
        y = np.zeros(sr * duration, dtype=np.float32)
    target_len = sr * duration
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]
    return y.astype(np.float32)

def audio_to_melspec(y, sr=CFG.sr):
    """Convert audio to log-mel spectrogram."""
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=CFG.n_mels, n_fft=CFG.n_fft,
        hop_length=CFG.hop_length, fmax=CFG.fmax
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    if mel_db.shape[1] < CFG.img_size[1]:
        mel_db = np.pad(mel_db, ((0, 0), (0, CFG.img_size[1] - mel_db.shape[1])))
    else:
        mel_db = mel_db[:, :CFG.img_size[1]]
    return mel_db.astype(np.float32)

def extract_features(y, sr=CFG.sr):
    """Extract handcrafted audio features for classical ML."""
    features = []
    # MFCCs (20 coefficients x mean + std = 40)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    features.extend(mfcc.mean(axis=1))
    features.extend(mfcc.std(axis=1))
    # Chroma (12 x 2 = 24)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    features.extend(chroma.mean(axis=1))
    features.extend(chroma.std(axis=1))
    # Spectral features
    features.append(librosa.feature.spectral_centroid(y=y, sr=sr).mean())
    features.append(librosa.feature.spectral_bandwidth(y=y, sr=sr).mean())
    features.append(librosa.feature.spectral_rolloff(y=y, sr=sr).mean())
    features.append(librosa.feature.spectral_contrast(y=y, sr=sr).mean())
    # Rhythm
    features.append(librosa.feature.tempo(y=y, sr=sr)[0])
    # Zero crossing rate
    features.append(librosa.feature.zero_crossing_rate(y).mean())
    # RMS energy
    features.append(librosa.feature.rms(y=y).mean())
    return np.array(features, dtype=np.float32)

In [ ]:
# ============================================================
# Cell 8: Build Training Catalog
# ============================================================
print("Building stem catalog...")
stem_catalog = {}  # genre -> list of song_dirs
for genre in CFG.genres:
    genre_dir = stems_path / genre
    songs = sorted([d for d in genre_dir.iterdir() if d.is_dir()])
    stem_catalog[genre] = songs
    print(f"  {genre}: {len(songs)} songs")

noise_files = sorted(list(noise_path.glob('*.wav')))
print(f"\nNoise files: {len(noise_files)}")

In [ ]:
# ============================================================
# Cell 9: Synthetic Mashup Generator
# ============================================================

def load_stem(song_dir, stem_name, sr=CFG.sr, duration=CFG.duration):
    """Load a single stem from a song directory."""
    path = song_dir / stem_name
    if path.exists():
        return load_audio(str(path), sr=sr, duration=duration)
    return np.zeros(sr * duration, dtype=np.float32)

def random_tempo_shift(y, sr, factor=None):
    """Apply random tempo change without pitch shift."""
    if factor is None:
        factor = np.random.uniform(0.85, 1.15)
    return librosa.effects.time_stretch(y, rate=factor)

def add_noise(y, noise_files, sr=CFG.sr, snr_db=None):
    """Add random noise at random position with random SNR."""
    if len(noise_files) == 0:
        return y
    noise_path_sel = random.choice(noise_files)
    noise = load_audio(str(noise_path_sel), sr=sr, duration=CFG.duration)
    if snr_db is None:
        snr_db = np.random.uniform(3, CFG.max_noise_snr)
    if len(noise) > 0:
        sig_power = np.mean(y ** 2) + 1e-10
        noise_power = np.mean(noise ** 2) + 1e-10
        scale = np.sqrt(sig_power / (noise_power * (10 ** (snr_db / 10))))
        max_offset = max(0, len(y) - len(noise))
        offset = random.randint(0, max_offset) if max_offset > 0 else 0
        y_out = y.copy()
        end = min(offset + len(noise), len(y_out))
        y_out[offset:end] += scale * noise[:end - offset]
        return y_out
    return y

def create_synthetic_mashup(genre, stem_catalog, noise_files,
                             sr=CFG.sr, duration=CFG.duration):
    """Create a mashup by mixing stems from DIFFERENT songs of the same genre."""
    songs = stem_catalog[genre]
    stem_names = ['drums.wav', 'vocals.wav', 'bass.wav', 'others.wav']
    selected_songs = random.choices(songs, k=len(stem_names))
    target_len = sr * duration
    mashup = np.zeros(target_len, dtype=np.float32)

    n_stems = random.randint(2, 4)
    chosen_stems = random.sample(list(zip(stem_names, selected_songs)), n_stems)

    for stem_name, song_dir in chosen_stems:
        stem = load_stem(song_dir, stem_name, sr=sr, duration=duration)
        if random.random() < 0.4:
            try:
                stem = random_tempo_shift(stem, sr)
                if len(stem) < target_len:
                    stem = np.pad(stem, (0, target_len - len(stem)))
                else:
                    stem = stem[:target_len]
            except Exception:
                pass
        gain = np.random.uniform(0.5, 1.5)
        mashup += gain * stem

    peak = np.max(np.abs(mashup)) + 1e-8
    mashup = mashup / peak

    if random.random() < CFG.noise_prob:
        n_noises = random.randint(1, 3)
        for _ in range(n_noises):
            mashup = add_noise(mashup, noise_files, sr=sr)
        peak = np.max(np.abs(mashup)) + 1e-8
        mashup = mashup / peak

    return mashup

In [ ]:
# ============================================================
# Cell 10: Generate Synthetic Training Data
# ============================================================
print("Generating synthetic training mashups...")
train_audios = []
train_labels = []

for genre in tqdm(CFG.genres, desc="Synthetic Mashups"):
    for i in range(CFG.n_mashups_per_genre):
        y = create_synthetic_mashup(genre, stem_catalog, noise_files)
        train_audios.append(y)
        train_labels.append(CFG.genre2idx[genre])

train_audios = np.array(train_audios)
train_labels = np.array(train_labels)
print(f"Synthetic mashups: {train_audios.shape[0]} samples")

# Add clean full mixes (all 4 stems from same song) for diversity
print("Adding clean full-mix samples...")
for genre in tqdm(CFG.genres, desc="Clean mixes"):
    for song_dir in stem_catalog[genre]:
        stems = []
        for s in ['drums.wav', 'vocals.wav', 'bass.wav', 'others.wav']:
            stems.append(load_stem(song_dir, s))
        mix = sum(stems) / 4.0
        peak = np.max(np.abs(mix)) + 1e-8
        mix = mix / peak
        train_audios = np.vstack([train_audios, mix.reshape(1, -1)])
        train_labels = np.append(train_labels, CFG.genre2idx[genre])

print(f"Total training set: {train_audios.shape[0]} samples")

In [ ]:
# ============================================================
# Cell 11: Convert Audio to Mel Spectrograms
# ============================================================
print("Converting training audio to mel spectrograms...")
train_specs = []
for y in tqdm(train_audios, desc="Mel spectrograms"):
    spec = audio_to_melspec(y)
    train_specs.append(spec)
train_specs = np.array(train_specs)
print(f"Spectrogram shape: {train_specs.shape}")

In [ ]:
# ============================================================
# Cell 12: Dataset & DataLoader
# ============================================================

class MelDataset(Dataset):
    def __init__(self, specs, labels=None, augment=False):
        self.specs = specs
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.specs)

    def __getitem__(self, idx):
        spec = self.specs[idx].copy()
        if self.augment:
            # Frequency masking
            if random.random() < 0.5:
                f = random.randint(0, 20)
                f0 = random.randint(0, max(0, spec.shape[0] - f))
                spec[f0:f0+f, :] = 0
            # Time masking
            if random.random() < 0.5:
                t = random.randint(0, 30)
                t0 = random.randint(0, max(0, spec.shape[1] - t))
                spec[:, t0:t0+t] = 0
            # Random gain
            if random.random() < 0.3:
                spec = spec * np.random.uniform(0.8, 1.2)
                spec = np.clip(spec, 0, 1)
        x = torch.FloatTensor(spec).unsqueeze(0)  # (1, n_mels, time)
        if self.labels is not None:
            return x, torch.LongTensor([self.labels[idx]])[0]
        return x

In [ ]:
# ============================================================
# Cell 13: Training Utilities
# ============================================================

def train_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    preds_all, labels_all = [], []
    for specs, labels in loader:
        specs, labels = specs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(specs)
        loss = F.cross_entropy(logits, labels, label_smoothing=0.1)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(labels)
        preds_all.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
    scheduler.step()
    avg_loss = total_loss / len(labels_all)
    f1 = f1_score(labels_all, preds_all, average='macro')
    return avg_loss, f1

@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    total_loss = 0
    preds_all, labels_all = [], []
    for specs, labels in loader:
        specs, labels = specs.to(device), labels.to(device)
        logits = model(specs)
        loss = F.cross_entropy(logits, labels)
        total_loss += loss.item() * len(labels)
        preds_all.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(labels_all)
    f1 = f1_score(labels_all, preds_all, average='macro')
    return avg_loss, f1, np.array(preds_all)

---
## Model 1: XGBoost (Classical ML Baseline)

In [ ]:
# ============================================================
# Cell 14: XGBoost on Handcrafted Features
# ============================================================
print("Extracting handcrafted features for training...")
X_feat = np.array([extract_features(y) for y in tqdm(train_audios, desc="Features")])
y_feat = train_labels.copy()

scaler = StandardScaler()
X_feat_scaled = scaler.fit_transform(X_feat)

# W&B run for XGBoost
run = wandb.init(project=W_PROJECT, name="xgboost-baseline", config={
    "model": "XGBClassifier", "n_estimators": 500, "max_depth": 8
})

xgb_model = XGBClassifier(
    n_estimators=500, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, tree_method='hist',
    eval_metric='mlogloss', random_state=CFG.seed
)

cv_scores = cross_val_score(xgb_model, X_feat_scaled, y_feat,
                            cv=5, scoring='f1_macro')
print(f"XGBoost CV Macro F1: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

wandb.log({"xgb_cv_f1_mean": cv_scores.mean(), "xgb_cv_f1_std": cv_scores.std()})

# Fit on full data
xgb_model.fit(X_feat_scaled, y_feat)

# Save
joblib.dump(xgb_model, 'xgb_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print("XGBoost model saved.")

wandb.finish()

---
## Model 2: CNN from Scratch

In [ ]:
# ============================================================
# Cell 15: CNN Model Definition
# ============================================================

class GenreClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.2),
            # Block 2
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.2),
            # Block 3
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.3),
            # Block 4
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

In [ ]:
# ============================================================
# Cell 16: CNN K-Fold Training with W&B
# ============================================================
skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
fold_models = []
oof_f1_scores = []

run = wandb.init(project=W_PROJECT, name="cnn-from-scratch", config={
    "model": "GenreClassifier_CNN", "epochs": CFG.epochs, "lr": CFG.lr,
    "batch_size": CFG.batch_size, "n_folds": CFG.n_folds,
    "n_mels": CFG.n_mels, "duration": CFG.duration
})

for fold, (train_idx, val_idx) in enumerate(skf.split(train_specs, train_labels)):
    print(f"\n{'='*50}")
    print(f"FOLD {fold + 1}/{CFG.n_folds}")
    print(f"{'='*50}")

    train_ds = MelDataset(train_specs[train_idx], train_labels[train_idx], augment=True)
    val_ds = MelDataset(train_specs[val_idx], train_labels[val_idx], augment=False)

    train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                              num_workers=CFG.num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CFG.batch_size * 2, shuffle=False,
                            num_workers=CFG.num_workers, pin_memory=True)

    model = GenreClassifier(num_classes=10).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs, eta_min=1e-6)

    best_f1 = 0
    patience_counter = 0

    for epoch in range(CFG.epochs):
        train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, scheduler, device)
        val_loss, val_f1, _ = validate(model, val_loader, device)

        wandb.log({
            f"fold{fold}/train_loss": train_loss, f"fold{fold}/train_f1": train_f1,
            f"fold{fold}/val_loss": val_loss, f"fold{fold}/val_f1": val_f1,
            "epoch": epoch
        })

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} F1: {train_f1:.4f} | "
                  f"Val Loss: {val_loss:.4f} F1: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), f'best_model_fold{fold}.pth')
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 8:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(torch.load(f'best_model_fold{fold}.pth'))
    fold_models.append(model)
    oof_f1_scores.append(best_f1)
    print(f"  Best Val F1: {best_f1:.4f}")

wandb.log({"cnn_mean_f1": np.mean(oof_f1_scores), "cnn_std_f1": np.std(oof_f1_scores)})
print(f"\nCNN Mean OOF F1: {np.mean(oof_f1_scores):.4f} +/- {np.std(oof_f1_scores):.4f}")
wandb.finish()

---
## Model 3: CRNN (CNN + Bidirectional GRU)

In [ ]:
# ============================================================
# Cell 17: CRNN Model Definition
# ============================================================

class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d((2, 2)), nn.Dropout(0.2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d((2, 2)), nn.Dropout(0.2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d((2, 1)), nn.Dropout(0.3),
        )
        self.freq_pool = nn.AdaptiveAvgPool2d((1, None))  # pool over freq, keep time
        self.gru = nn.GRU(128, 128, num_layers=2, batch_first=True,
                          dropout=0.3, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(256, 64),  # 128*2 for bidirectional
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.cnn(x)                          # (B, 128, F', T')
        x = self.freq_pool(x)                    # (B, 128, 1, T')
        x = x.squeeze(2).permute(0, 2, 1)        # (B, T', 128)
        out, _ = self.gru(x)                      # (B, T', 256)
        x = out[:, -1, :]                         # last time step
        return self.classifier(x)

In [ ]:
# ============================================================
# Cell 18: CRNN K-Fold Training with W&B
# ============================================================
run = wandb.init(project=W_PROJECT, name="crnn-bigru", config={
    "model": "CRNN_BiGRU", "epochs": CFG.epochs, "lr": CFG.lr
})

skf2 = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
crnn_models = []
crnn_f1_scores = []

for fold, (train_idx, val_idx) in enumerate(skf2.split(train_specs, train_labels)):
    print(f"\nCRNN FOLD {fold + 1}/{CFG.n_folds}")

    train_ds = MelDataset(train_specs[train_idx], train_labels[train_idx], augment=True)
    val_ds = MelDataset(train_specs[val_idx], train_labels[val_idx], augment=False)
    train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                              num_workers=CFG.num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CFG.batch_size * 2, shuffle=False,
                            num_workers=CFG.num_workers, pin_memory=True)

    model = CRNN(num_classes=10).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs, eta_min=1e-6)

    best_f1 = 0
    patience_counter = 0
    for epoch in range(CFG.epochs):
        train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, scheduler, device)
        val_loss, val_f1, _ = validate(model, val_loader, device)
        wandb.log({f"crnn_fold{fold}/train_loss": train_loss, f"crnn_fold{fold}/val_f1": val_f1})

        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1:02d} | Val F1: {val_f1:.4f}")
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), f'crnn_fold{fold}.pth')
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 8:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(torch.load(f'crnn_fold{fold}.pth'))
    crnn_models.append(model)
    crnn_f1_scores.append(best_f1)
    print(f"  Best Val F1: {best_f1:.4f}")

wandb.log({"crnn_mean_f1": np.mean(crnn_f1_scores)})
print(f"\nCRNN Mean F1: {np.mean(crnn_f1_scores):.4f} +/- {np.std(crnn_f1_scores):.4f}")
wandb.finish()

---
## Model 4: Pretrained Audio Spectrogram Transformer (AST)

In [ ]:
# ============================================================
# Cell 19: AST Model Definition
# ============================================================
from transformers import ASTForAudioClassification, ASTFeatureExtractor

AST_MODEL = "MIT/ast-finetuned-audioset-10-10-0.4593"
ast_extractor = ASTFeatureExtractor.from_pretrained(AST_MODEL)

class ASTDataset(Dataset):
    def __init__(self, audios, labels=None, sr=CFG.sr):
        self.audios = audios
        self.labels = labels
        self.sr = sr

    def __len__(self):
        return len(self.audios)

    def __getitem__(self, idx):
        inputs = ast_extractor(self.audios[idx], sampling_rate=self.sr,
                               return_tensors="pt", padding="max_length",
                               max_length=1024, truncation=True)
        pixel_values = inputs['input_values'].squeeze(0)
        if self.labels is not None:
            return pixel_values, torch.tensor(self.labels[idx], dtype=torch.long)
        return pixel_values

class ASTClassifier(nn.Module):
    def __init__(self, num_classes=10, model_name=AST_MODEL):
        super().__init__()
        self.ast = ASTForAudioClassification.from_pretrained(
            model_name, num_labels=num_classes, ignore_mismatched_sizes=True
        )

    def forward(self, x):
        return self.ast(x).logits

In [ ]:
# ============================================================
# Cell 20: AST Fine-Tuning with W&B
# ============================================================
run = wandb.init(project=W_PROJECT, name="ast-finetuned", config={
    "model": "AST-audioset", "epochs": 10, "lr": 5e-5, "batch_size": 16
})

ast_model = ASTClassifier(num_classes=10).to(device)

# Freeze early layers, fine-tune last 4 layers + classifier
for param in ast_model.ast.audio_spectrogram_transformer.parameters():
    param.requires_grad = False
for layer in ast_model.ast.audio_spectrogram_transformer.encoder.layer[-4:]:
    for param in layer.parameters():
        param.requires_grad = True
for param in ast_model.ast.classifier.parameters():
    param.requires_grad = True

ast_optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, ast_model.parameters()),
    lr=5e-5, weight_decay=1e-4
)
ast_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(ast_optimizer, T_max=10)

# 80/20 split (AST is expensive to train)
tr_idx, vl_idx = train_test_split(
    range(len(train_audios)), test_size=0.2,
    stratify=train_labels, random_state=CFG.seed
)

ast_train_ds = ASTDataset(train_audios[tr_idx], train_labels[tr_idx])
ast_val_ds = ASTDataset(train_audios[vl_idx], train_labels[vl_idx])
ast_train_loader = DataLoader(ast_train_ds, batch_size=16, shuffle=True,
                              num_workers=2, pin_memory=True)
ast_val_loader = DataLoader(ast_val_ds, batch_size=32, shuffle=False,
                            num_workers=2, pin_memory=True)

best_ast_f1 = 0
for epoch in range(10):
    ast_model.train()
    total_loss = 0
    for x, y in tqdm(ast_train_loader, desc=f"AST Epoch {epoch+1}"):
        x, y = x.to(device), y.to(device)
        ast_optimizer.zero_grad()
        logits = ast_model(x)
        loss = F.cross_entropy(logits, y, label_smoothing=0.1)
        loss.backward()
        ast_optimizer.step()
        total_loss += loss.item() * len(y)
    ast_scheduler.step()

    # Validate
    ast_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in ast_val_loader:
            x = x.to(device)
            preds = ast_model(x).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y.numpy())

    val_f1 = f1_score(all_labels, all_preds, average='macro')
    wandb.log({"ast/train_loss": total_loss / len(tr_idx), "ast/val_f1": val_f1})
    print(f"  AST Epoch {epoch+1} | Val F1: {val_f1:.4f}")

    if val_f1 > best_ast_f1:
        best_ast_f1 = val_f1
        torch.save(ast_model.state_dict(), 'ast_best.pth')

ast_model.load_state_dict(torch.load('ast_best.pth'))
print(f"\nBest AST Val F1: {best_ast_f1:.4f}")
wandb.finish()

---
## Save All Artifacts for Inference Notebook

In [ ]:
# ============================================================
# Cell 21: Save All Model Artifacts
# ============================================================
# CNN fold models
for i, m in enumerate(fold_models):
    torch.save(m.state_dict(), f'cnn_fold{i}.pth')
    print(f"Saved cnn_fold{i}.pth")

# CRNN fold models
for i, m in enumerate(crnn_models):
    torch.save(m.state_dict(), f'crnn_fold{i}.pth')
    print(f"Saved crnn_fold{i}.pth")

# AST model (already saved as ast_best.pth)
print("Saved ast_best.pth")

# XGBoost + scaler (already saved)
print("Saved xgb_model.pkl, scaler.pkl")

# Save AST pretrained weights for offline inference
ast_extractor.save_pretrained('./ast_pretrained')
ast_model.ast.save_pretrained('./ast_pretrained')
print("Saved AST pretrained weights to ./ast_pretrained/")

print("\n" + "="*50)
print("IMPORTANT: After this notebook finishes:")
print("1. Go to Output tab")
print("2. Click 'New Dataset' to upload all .pth, .pkl files")
print("3. Also upload ./ast_pretrained/ folder as a separate dataset")
print("4. Use these datasets in your Inference notebook")
print("="*50)

---
## Model Comparison & Error Analysis

In [ ]:
# ============================================================
# Cell 22: Model Comparison
# ============================================================
results = {
    "XGBoost": cv_scores.mean(),
    "CNN": np.mean(oof_f1_scores),
    "CRNN": np.mean(crnn_f1_scores),
    "AST (pretrained)": best_ast_f1,
}

print("=" * 45)
print("MODEL COMPARISON (Validation Macro F1)")
print("=" * 45)
for name, score in sorted(results.items(), key=lambda x: -x[1]):
    bar = chr(9608) * int(score * 40)
    print(f"  {name:20s} | {score:.4f} | {bar}")

# Log comparison to W&B
run = wandb.init(project=W_PROJECT, name="model-comparison")
wandb.log(results)
comparison_table = wandb.Table(
    columns=["Model", "Val_Macro_F1"],
    data=[[k, v] for k, v in results.items()]
)
wandb.log({"comparison": comparison_table})
wandb.finish()

# Bar chart
plt.figure(figsize=(8, 5))
names = list(results.keys())
scores = list(results.values())
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']
plt.barh(names, scores, color=colors)
plt.xlabel('Macro F1 Score')
plt.title('Model Comparison')
plt.xlim(0, 1)
for i, v in enumerate(scores):
    plt.text(v + 0.01, i, f'{v:.4f}', va='center')
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Cell 23: Error Analysis (Confusion Matrix & Per-Genre F1)
# ============================================================

# Get validation predictions from CNN fold 0
fold0_splits = list(skf.split(train_specs, train_labels))
val_idx_f0 = fold0_splits[0][1]

val_ds_f0 = MelDataset(train_specs[val_idx_f0], train_labels[val_idx_f0], augment=False)
val_loader_f0 = DataLoader(val_ds_f0, batch_size=CFG.batch_size * 2, shuffle=False,
                           num_workers=CFG.num_workers)

_, _, val_preds_f0 = validate(fold_models[0], val_loader_f0, device)
val_true_f0 = train_labels[val_idx_f0]

# Confusion Matrix
cm = confusion_matrix(val_true_f0, val_preds_f0)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CFG.genres, yticklabels=CFG.genres)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('CNN Fold-0 Confusion Matrix')
plt.tight_layout(); plt.show()

# Per-class F1 report
print("\nPer-Genre Classification Report (CNN Fold-0):")
print(classification_report(val_true_f0, val_preds_f0, target_names=CFG.genres))

# Genre-wise F1
from sklearn.metrics import f1_score as f1_fn
genre_f1 = {}
for i, g in enumerate(CFG.genres):
    mask_true = (val_true_f0 == i)
    mask_pred = (val_preds_f0 == i)
    tp = (mask_true & mask_pred).sum()
    fp = (~mask_true & mask_pred).sum()
    fn = (mask_true & ~mask_pred).sum()
    prec = tp / (tp + fp + 1e-8)
    rec = tp / (tp + fn + 1e-8)
    f1 = 2 * prec * rec / (prec + rec + 1e-8)
    genre_f1[g] = f1

plt.figure(figsize=(10, 5))
plt.bar(genre_f1.keys(), genre_f1.values(), color='steelblue')
plt.ylabel('F1 Score'); plt.title('Per-Genre F1 (CNN Fold-0)')
plt.xticks(rotation=45); plt.ylim(0, 1)
plt.axhline(y=np.mean(list(genre_f1.values())), color='red', linestyle='--', label='Mean')
plt.legend(); plt.tight_layout(); plt.show()

---
## Inference: Load Test Data & Generate Submission

In [ ]:
# ============================================================
# Cell 24: Load & Prepare Test Data
# ============================================================
test_df = pd.read_csv(Path(CFG.base_path) / 'test.csv')
mashups_path = Path(CFG.base_path) / 'mashups'

# Auto-detect filename pattern
if 'filename' in test_df.columns:
    test_files = [mashups_path / row['filename'] for _, row in test_df.iterrows()]
else:
    sample_id = test_df['id'].iloc[0]
    if (mashups_path / f"{sample_id}.wav").exists():
        test_files = [mashups_path / f"{row['id']}.wav" for _, row in test_df.iterrows()]
    elif (mashups_path / f"mashup_{sample_id}.wav").exists():
        test_files = [mashups_path / f"mashup_{row['id']}.wav" for _, row in test_df.iterrows()]
    else:
        all_wavs = sorted(mashups_path.glob('*.wav'))
        test_files = all_wavs[:len(test_df)]
        print(f"Warning: Auto-mapped {len(test_files)} files by sorted order")

print(f"Test files: {len(test_files)}")
print(f"First 3: {[f.name for f in test_files[:3]]}")

# Load all test audio once
print("\nLoading test audio...")
test_audios = np.array([load_audio(str(f)) for f in tqdm(test_files, desc="Loading")])

# Mel spectrograms for CNN/CRNN
print("Computing mel spectrograms...")
test_specs = np.array([audio_to_melspec(y) for y in tqdm(test_audios, desc="Mel specs")])

# Handcrafted features for XGBoost
print("Extracting handcrafted features...")
X_test_feat = np.array([extract_features(y) for y in tqdm(test_audios, desc="Features")])
X_test_scaled = scaler.transform(X_test_feat)

print(f"\nSpecs shape: {test_specs.shape} | Features shape: {X_test_feat.shape}")

In [ ]:
# ============================================================
# Cell 25: Run Inference on All Models
# ============================================================

@torch.no_grad()
def get_probs(models, loader, device):
    """Get averaged softmax probabilities from an ensemble of models."""
    all_probs = []
    for model in models:
        model.eval()
        probs = []
        for batch in loader:
            x = batch[0] if isinstance(batch, (list, tuple)) else batch
            x = x.to(device)
            logits = model(x)
            probs.append(F.softmax(logits, dim=1).cpu().numpy())
        all_probs.append(np.concatenate(probs, axis=0))
    return np.mean(all_probs, axis=0)

# Mel spectrogram test loader (shared by CNN and CRNN)
test_mel_ds = MelDataset(test_specs, labels=None, augment=False)
test_mel_loader = DataLoader(test_mel_ds, batch_size=CFG.batch_size * 2,
                              shuffle=False, num_workers=CFG.num_workers)

# 1. CNN predictions
print("Running CNN inference...")
cnn_probs = get_probs(fold_models, test_mel_loader, device)
print(f"  CNN probs shape: {cnn_probs.shape}")

# 2. CRNN predictions
print("Running CRNN inference...")
crnn_probs = get_probs(crnn_models, test_mel_loader, device)
print(f"  CRNN probs shape: {crnn_probs.shape}")

# 3. AST predictions
print("Running AST inference...")
ast_test_ds = ASTDataset(test_audios, labels=None)
ast_test_loader = DataLoader(ast_test_ds, batch_size=32, shuffle=False, num_workers=2)
ast_probs = get_probs([ast_model], ast_test_loader, device)
print(f"  AST probs shape: {ast_probs.shape}")

# 4. XGBoost predictions
print("Running XGBoost inference...")
xgb_probs = xgb_model.predict_proba(X_test_scaled)
print(f"  XGBoost probs shape: {xgb_probs.shape}")

print("\nAll model predictions complete.")

In [ ]:
# ============================================================
# Cell 26: Weighted Ensemble & Submission
# ============================================================

# Weights (tune based on validation F1)
W_CNN  = 0.25
W_CRNN = 0.20
W_AST  = 0.40
W_XGB  = 0.15

final_probs = (W_CNN * cnn_probs + W_CRNN * crnn_probs +
               W_AST * ast_probs + W_XGB * xgb_probs)

final_preds = final_probs.argmax(axis=1)
predicted_genres = [CFG.idx2genre[p] for p in final_preds]

# Build submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'genre': predicted_genres
})

# Sanity checks
assert len(submission) == len(test_df), f"Length mismatch: {len(submission)} vs {len(test_df)}"
assert set(submission['genre'].unique()).issubset(set(CFG.genres)), "Invalid genre label found!"

print(f"Total predictions: {len(submission)}")
print(f"\nPrediction distribution:")
print(submission['genre'].value_counts().to_string())

submission.to_csv('submission.csv', index=False)
print(f"\nsubmission.csv saved ({len(submission)} rows)")
print(submission.head(10))

In [ ]:
# ============================================================
# Cell 27: Confidence Analysis
# ============================================================

max_confidence = final_probs.max(axis=1)
print(f"Mean confidence: {max_confidence.mean():.4f}")
print(f"Low confidence (<0.5) samples: {(max_confidence < 0.5).sum()} / {len(max_confidence)}")

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.hist(max_confidence, bins=50, color='steelblue', edgecolor='black')
plt.xlabel('Prediction Confidence'); plt.ylabel('Count')
plt.title('Ensemble Confidence Distribution')

plt.subplot(1, 2, 2)
genre_conf = {}
for g in CFG.genres:
    mask = np.array(predicted_genres) == g
    if mask.sum() > 0:
        genre_conf[g] = max_confidence[mask].mean()
plt.barh(list(genre_conf.keys()), list(genre_conf.values()), color='steelblue')
plt.xlabel('Mean Confidence'); plt.title('Confidence by Predicted Genre')
plt.tight_layout(); plt.show()

# Model agreement
cnn_pred = cnn_probs.argmax(1)
crnn_pred = crnn_probs.argmax(1)
ast_pred = ast_probs.argmax(1)

all_agree = (cnn_pred == crnn_pred) & (crnn_pred == ast_pred)
print(f"\nAll 3 DL models agree: {all_agree.sum()}/{len(all_agree)} ({all_agree.mean()*100:.1f}%)")

# Where models disagree most
disagree_idx = np.where(~all_agree)[0]
if len(disagree_idx) > 0:
    print(f"\nSample disagreements (first 5):")
    for idx in disagree_idx[:5]:
        print(f"  ID {test_df['id'].iloc[idx]}: "
              f"CNN={CFG.idx2genre[cnn_pred[idx]]} | "
              f"CRNN={CFG.idx2genre[crnn_pred[idx]]} | "
              f"AST={CFG.idx2genre[ast_pred[idx]]} | "
              f"Ensemble={predicted_genres[idx]}")

In [ ]:
# ============================================================
# Cell 28: Final Summary
# ============================================================
print("\n" + "="*60)
print("  TRAINING COMPLETE - FINAL SUMMARY")
print("="*60)
print(f"\n  Models trained:")
print(f"    1. XGBoost (Classical ML)    : CV F1 = {cv_scores.mean():.4f}")
print(f"    2. CNN (From Scratch)        : OOF F1 = {np.mean(oof_f1_scores):.4f}")
print(f"    3. CRNN (CNN+BiGRU)          : OOF F1 = {np.mean(crnn_f1_scores):.4f}")
print(f"    4. AST (Pretrained)          : Val F1 = {best_ast_f1:.4f}")
print(f"\n  Ensemble weights: CNN={W_CNN} CRNN={W_CRNN} AST={W_AST} XGB={W_XGB}")
print(f"  Test predictions: {len(submission)}")
print(f"  Submission saved: submission.csv")
print(f"\n  Artifacts saved:")
print(f"    - cnn_fold{{0-4}}.pth")
print(f"    - crnn_fold{{0-4}}.pth")
print(f"    - ast_best.pth")
print(f"    - xgb_model.pkl, scaler.pkl")
print(f"    - ast_pretrained/ (for offline inference)")
print(f"\n  W&B project: {W_PROJECT}")
print("="*60)